# Semana 4 · Clase 1 — Primera escena 3D real (pythreejs)

**Electiva Profesional I (El Metaverso) · 2026-II**

La semana pasada construyó la galería **como un conjunto datos**: una lista de diccionarios, cada uno con una `posicion` (una tupla `(x, y, z)`) que no se vió en realidad. Hoy esa posición dejar de ser solo un
dato: hoy **usted arma la sala y cuelga las obras donde esa tupla dice**, con **pythreejs**, la librería
que dibuja escenas 3D dentro de Jupyter.

En el camino usa **numpy para transformar la geometría** — trasladar, rotar, escalar — y se enfrenta a un nuevo reto: dos transformaciones que, aplicadas en **el orden equivocado**, dan un resultado
que se ve razonable pero está mal, **sin ningún error de ejecución de `python`**. 

Ejecute de arriba abajo. En **`[Trabajo del estudiante]`** no hay código de 
partida — se apoya en el patrón que la celda anterior acaba de mostrar. 


## 0. Entorno

Mismo entorno `mv2026` de las semanas 1 a 3. 


In [1]:
import sys
import numpy as np

print('Ejecutable (debe mencionar mv2026):', sys.executable)
print('numpy:', np.__version__)

# pythreejs es lo unico nuevo hoy. Si por alguna razon no cargara en un equipo,
# el cuaderno sigue funcionando -toda la parte de numpy no depende de el- y ese
# equipo puede seguir el plan B: la version WebXR de manana en el navegador.
try:
    import pythreejs as p3
    TENGO_3D = True
    print('pythreejs: OK')
except Exception as e:
    TENGO_3D = False
    print('pythreejs: no disponible ->', e)


Ejecutable (debe mencionar mv2026): C:\Users\loghan\miniconda3\envs\mv2026\python.exe
numpy: 2.5.1
pythreejs: OK


---
## 1. La sala: `Mesh`, `geometry` y `material`

Toda malla (`mesh`) en pythreejs se arma con tres piezas: una **geometría** (la forma: caja,
esfera, plano), un **material** (de qué está hecha: color, si brilla) y una **posición** (dónde
va). Ya vio `SphereGeometry` en la semana 1; hoy suma `PlaneGeometry` (un plano, para el piso) y
`BoxGeometry` (una caja, para la pared y las obras — un cuadro es, geométricamente, una caja muy
delgada).

In [2]:
piso = p3.Mesh(
    geometry=p3.PlaneGeometry(width=40, height=20),
    material=p3.MeshStandardMaterial(color='#b9b0a0', side='DoubleSide'),
    position=[5.0, 0.0, 0.0])
piso.rotateX(-np.pi / 2)  

pared = p3.Mesh(
    geometry=p3.BoxGeometry(width=14, height=4, depth=0.2),
    material=p3.MeshStandardMaterial(color='#e4ded0'),
    position=[0.0, 2.0, -4.0])

print('piso y pared construidos. Todavia no se ve nada: falta armar la Scene completa (seccion 6).')


piso y pared construidos. Todavia no se ve nada: falta armar la Scene completa (seccion 6).


---
## 2. Colgar la primera obra: la misma posición de la semana pasada

Esta es, **exactamente**, la posición de `"El Reflejo Digital"` que quedó guardada en
`galeria[0]["posicion"]` la semana pasada: `(2.0, 0.0, 5.0)`. La semana pasada esa tupla era solo
un dato en un diccionario. Hoy es literalmente **dónde va a aparecer la obra en la escena**.

In [3]:
posicion_reflejo = np.array([2.0, 0.0, 5.0])  # igual a la semana 3, sin cambios

obra_reflejo = p3.Mesh(
    geometry=p3.BoxGeometry(width=1.0, height=1.0, depth=0.1),
    material=p3.MeshStandardMaterial(color='#3b7dd8'),
    position=posicion_reflejo.tolist())

print('El Reflejo Digital construida en', posicion_reflejo)


El Reflejo Digital construida en [2. 0. 5.]


**`# TODO 1` — Ahora usted.** La semana pasada, en `galeria`, también guardó estas dos posiciones
(cópielas tal cual, no las cambie):

- `"Fragmentos"` → `(-3.0, 1.5, 2.0)`
- `"Eco"` → `(5.0, 0.0, 1.0)`

Siguiendo **exactamente** el patrón de la celda anterior (un `np.array` con la posición, un
`p3.Mesh` con `BoxGeometry` y `MeshStandardMaterial`), construya `obra_fragmentos` y `obra_eco`.
Elija dos colores distintos entre sí y distintos del azul de `"El Reflejo Digital"`.


In [4]:
# TODO 1: construya obra_fragmentos y obra_eco, con las posiciones de la semana pasada.
posicion_fragmento = np.array([-3.0, 1.5, 2.0])
obra_fragmento = p3.Mesh(
    geometry = p3.BoxGeometry(width=1.0, height=1.0, depth=0.1),
    material = p3.MeshStandardMaterial(color="#793791"),
    position = posicion_fragmento.tolist()
)
print('Fragmento esta construida en', posicion_fragmento)
posicion_eco = np.array([5.0, 0.0, 1.0])
obra_fragmento = p3.Mesh(
    geometry = p3.BoxGeometry(width=1.0, height=1.0, depth=0.1),
    material = p3.MeshStandardMaterial(color="#375191"),
    position = posicion_eco.tolist()
)
print('Eco esta construida en', posicion_eco)

Fragmento esta construida en [-3.   1.5  2. ]
Eco esta construida en [5. 0. 1.]


---
### 🅢 Punto de encuentro 1 — mesh, geometría, material, posición

- **Geometría** = la forma (`BoxGeometry`, `PlaneGeometry`, `SphereGeometry` de la S1).
- **Material** = de qué está hecha (acá, solo el color).
- **`Mesh`** = geometría + material, ya lista para colocar.
- **`position=[x, y, z]`** = la **traslación**: mover algo desde el origen `(0,0,0)` hasta donde
  debe estar. La tupla `posicion` de la semana pasada **era** exactamente esto — un dato esperando
  a convertirse en geometría.


---
## 3. Una obra nueva: rotar para ubicar

Sume una cuarta obra a la galería: una escultura, `"Punto de Fuga"`, que va en el **centro de la
sala**, sobre un círculo imaginario de radio 3 alrededor del centro — la misma idea de la órbita
de la semana 1 (`posicion_luna`), pero esta vez aplicada a decorar una sala, no a un planeta.

Recuerde la lección de la semana 1: `numpy` trabaja en **radianes**, no en grados. `np.radians(...)`
convierte antes de usar `np.sin`/`np.cos`.

In [5]:
def posicion_en_circulo(angulo_grados, radio, altura=0.0):
    a = np.radians(angulo_grados)          # grados -> radianes, ANTES de sin/cos (leccion de la S1)
    x = radio * np.sin(a)
    z = radio * np.cos(a)
    return np.array([x, altura, z])

posicion_escultura = posicion_en_circulo(45, radio=3.0)
print('Punto de Fuga en:', posicion_escultura.round(2))

escultura = p3.Mesh(
    geometry=p3.SphereGeometry(radius=0.5, widthSegments=20, heightSegments=20),
    material=p3.MeshStandardMaterial(color='#9b59b6'),
    position=posicion_escultura.tolist())


Punto de Fuga en: [2.12 0.   2.12]


---
## 4. El choque de hoy: el orden de las transformaciones

La galería decidió mudar las **tres obras originales** (`El Reflejo Digital`, `Fragmentos`, `Eco`)
a una sala nueva: **la mitad de grande** (`factor_escala = 0.5`) y ubicada **en otra parte del
mundo**, 12 unidades más allá en `x` (`destino_sala`).

Alguien escribe la función así — trasladar primero, escalar después — porque en el enunciado
"mudar y encoger" el verbo "mudar" viene primero:


In [6]:
factor_escala = 0.5
destino_sala = np.array([12.0, 0.0, 0.0])

posiciones_obras = {
    'El Reflejo Digital': posicion_reflejo,
    'Fragmentos': posicion_fragmento,
    'Eco': posicion_eco,
}

def mover_ingenuo(posicion):
    return (posicion + destino_sala) * factor_escala

print('Resultado (trasladar, LUEGO escalar):')
for nombre, pos in posiciones_obras.items():
    print(f'  {nombre:<20} -> {mover_ingenuo(pos).round(2)}')


Resultado (trasladar, LUEGO escalar):
  El Reflejo Digital   -> [7.  0.  2.5]
  Fragmentos           -> [4.5  0.75 1.  ]
  Eco                  -> [8.5 0.  0.5]


**`# TODO 2`** — La sala nueva debía quedar **alrededor de `x = 12`** (más 0, menos 0, según cada
obra). Mire los resultados: ninguno llegó ni cerca. **No hay ningún error** — el código corrió y
dio números. ¿Qué cree que salió mal? Escriba su intuición antes de seguir. _(reemplace esta línea)_

Al ejecutar (posicion + destino_sala) * factor_escala, la multiplicación por factor_escala (0.5) se aplica a toda la suma, reduciendo a la mitad tanto las coordenadas de la obra como el vector de traslación destino_sala ([12.0, 0.0, 0.0]). Como resultado, la sala entera fue desplazada únicamente hasta x = 6.0 en lugar de x = 12.0. 

### La corrección: escalar primero, trasladar después

El problema: `(posicion + destino_sala) * factor_escala` también multiplica **el destino** por
`0.5` — la sala nueva terminó a mitad de camino, no en `x = 12`. La sala se debe encoger **primero**
(sobre su propio centro, antes de moverla) y **después** desplazarse a su lugar final:


In [7]:
def mover_correcto(posicion):
    return (posicion * factor_escala) + destino_sala

print('Resultado (escalar, LUEGO trasladar):')
posiciones_nuevas = {}
for nombre, pos in posiciones_obras.items():
    nueva = mover_correcto(pos)
    posiciones_nuevas[nombre] = nueva
    print(f'  {nombre:<20} -> {nueva.round(2)}')


Resultado (escalar, LUEGO trasladar):
  El Reflejo Digital   -> [13.   0.   2.5]
  Fragmentos           -> [10.5   0.75  1.  ]
  Eco                  -> [14.5  0.   0.5]


**`# TODO 3`** — Ahora sí, los tres valores de `x` quedan alrededor de `12`. En sus palabras: ¿Por
qué escalar **antes** de trasladar evita que el destino también se encoja? ¿En qué otra situación
del curso (piense en la semana 3) una operación aplicada en el orden equivocado cambió el
resultado sin lanzar ningún error? _(reemplace esta línea)_

Escalar antes de trasladar ((posicion * factor_escala) + destino_sala) restringe la reducción de tamaño a la geometría de la obra con respecto a su origen local. Al sumar destino_sala después, el vector de posición final no es afectado por el factor 0.5 y sitúa la sala correctamente alrededor de x = 12.0


---
### 🅢 Punto de encuentro 2 — traslación, rotación y escala, en orden

- **Traslación** = sumar un vector (`posicion + destino`) — mueve sin cambiar tamaño ni forma.
- **Rotación** = combinar `sin`/`cos` de un ángulo en **radianes** — mueve alrededor de un centro.
- **Escala** = multiplicar por un número — cambia tamaño (y, si se aplica después de trasladar, **también
  encoge la traslación misma** — el choque de hoy).
- **Regla práctica:** cuando combine escala y traslación, **escale primero** (relativo al propio
  origen del objeto), **traslade después** (al lugar final). El mismo cuidado con el **orden** que
  ya vio con `crear_sala` en la semana 3, aplicado ahora a geometría en vez de a listas.


---
## 5. Cámara y luces

Sin cámara no hay desde dónde mirar; sin luz, `MeshStandardMaterial` se ve negro (reacciona a la
luz, no brilla solo). Igual que en la semana 1: una luz direccional (como el sol) y una luz
ambiente (para que las sombras no queden negro puro).


In [8]:
camara = p3.PerspectiveCamera(position=[10, 8, 14], fov=50)
luz = p3.DirectionalLight(color='white', intensity=1.0, position=[5, 8, 5])
ambiente = p3.AmbientLight(intensity=0.4)


---
## 6. Armar la escena completa

Una `Scene` es la lista de todo lo que existe: mallas, cámara y luces. El `Renderer` es lo que
efectivamente dibuja esa escena en el cuaderno.

**`[Trabajo del estudiante — # TODO 4]`** Construya un `Mesh` para cada obra en `posiciones_nuevas`
(mismo patrón de la sección 2: `BoxGeometry` + `MeshStandardMaterial` + `position`) y arme una escena
`Scene` con **todo** lo de hoy: `piso`, `pared`, las tres obras reubicadas, `escultura`, `camara`,
`luz` y `ambiente`. Termine con el `Renderer` (mismo patrón de la semana 1).


In [9]:
# TODO 4: un Mesh por cada obra de posiciones_nuevas y la Scene + Renderer con todo lo de hoy.

colores_obras = {
    'El Reflejo Digital': '#3b7dd8',
    'Fragmentos': '#793791',
    'Eco': '#375191'
}

obras_reubicadas = []
for nombre, pos in posiciones_nuevas.items():
    obra = p3.Mesh(
        geometry=p3.BoxGeometry(width=1.0 * factor_escala, height=1.0 * factor_escala, depth=0.1 * factor_escala),
        material=p3.MeshStandardMaterial(color=colores_obras[nombre]),
        position=pos.tolist()
    )
    obras_reubicadas.append(obra)

elementos_escena = [piso, pared, escultura, camara, luz, ambiente] + obras_reubicadas
escena = p3.Scene(children=elementos_escena)

renderer = p3.Renderer(
    camera=camara,
    scene=escena,
    controls=[p3.OrbitControls(controlling=camara)],
    width=800,
    height=500
)

display(renderer)

Renderer(camera=PerspectiveCamera(position=(10.0, 8.0, 14.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, …

---
## Cierre

1. Revise que la escena tenga las **siete** piezas: piso, pared, las tres obras reubicadas, la
   escultura, y que la cámara y las luces estén en la `Scene` (si no, no se ve nada, aunque no dé
   error).
